In [ ]:
try:
    import claveles  # noqa: F401
except ImportError:
    %pip install claveles

**Note for Google Colab Users:**
During installation, Colab may display a dependency conflict error regarding `protobuf` (`ERROR: pip's dependency resolver...`). This occurs due to pre-installed Google packages in the Colab environment. 

As long as you see `Successfully installed ...` at the end of the log, the required packages are properly installed. **Please ignore the error and continue running the rest of the notebook.**

$$
\newcommand{\bra}[1]{{\left\langle{#1}\right|}}
\newcommand{\ket}[1]{{\left|{#1}\right\rangle}}
\newcommand{\Bra}[1]{{\Big\langle{#1}\Big|}}
\newcommand{\Ket}[1]{{\Big|{#1}\Big\rangle}}
\newcommand{\braket}[1]{{\left\langle{#1}\right\rangle}}
\newcommand{\dag}{\dagger}
$$

# Fourier Transform via Phase Rotation

## Introduction

In continuous-variable (CV) quantum computing, a quantum state can be represented by its wave function in either the $x$-quadrature or the $p$-quadrature basis. These two representations are fundamentally connected by a *Fourier transform*. In this tutorial, we demonstrate that a $\hat{R}(-\pi/2)$ phase rotation (a 90-degree rotation in phase space) implements this Fourier transform between the position and momentum quadratures. Taking the superposition state $\ket{\psi} \propto \ket{0} + \ket{1} + \ket{2}$ as an example, we compute its Fourier transform analytically using closed-form Fock wave functions and confirm that a homodyne measurement of the $x$-quadrature after applying $\hat{R}(-\pi/2)$ successfully reproduces the predicted distribution.

## Background

A quantum state $\ket{\psi}$ can be characterized either by its $x$-quadrature wave function $\psi(x) = \braket{x | \psi}$ or by its $p$-quadrature wave function $\tilde\psi(p) = \braket{p | \psi}$. Using the completeness of the quadrature eigenstates $\hat{1} = \int_{-\infty}^\infty dx\ket{x}\bra{x}=\int_{-\infty}^\infty dp\ket{p}\bra{p}$ and their inner products being $\braket{x|p}=\frac{1}{\sqrt{2\pi\hbar}} e^{ixp/\hbar}$, one obtains

$$\begin{aligned} 
    \psi(x) &= \braket{x | \psi}\\
    &= \int_{-\infty}^\infty dp\, \braket{x | p}\braket{p |\psi}\\
    &= \int_{-\infty}^\infty dp\, \frac{1}{\sqrt{2\pi\hbar}} e^{ixp/\hbar}\braket{p |\psi}\\
    &= \frac{1}{\sqrt{2\pi\hbar}}\int_{-\infty}^\infty dp\,  e^{ixp/\hbar}\tilde\psi(p)
    \end{aligned}
$$

and

$$
    \begin{aligned} 
    \tilde\psi(p) &= \braket{p | \psi}\\
    &= \int_{-\infty}^\infty dx\, \braket{p | x}\braket{x |\psi}\\
    &= \int_{-\infty}^\infty dx\, \frac{1}{\sqrt{2\pi\hbar}} e^{-ixp/\hbar}\braket{x |\psi}\\
    &= \frac{1}{\sqrt{2\pi\hbar}}\int_{-\infty}^\infty dx\,  e^{-ixp/\hbar}\psi(x)
    \end{aligned}.
$$

Thus, these two representations are related to one another via a Fourier transform:

$$
\tilde\psi(p) = \mathcal{F}[\psi](p), \, \psi(x) = \mathcal{F}^{-1}[\tilde\psi](x)
$$

in which we defined $\mathcal{F}[f](s) = \frac{1}{\sqrt{2\pi\hbar}}\int_{-\infty}^\infty dt\,  e^{-ist/\hbar}f(t)$. The *phase rotation gate* $\hat{R}(\theta)$ is the unitary operator corresponding to a rotation in the $(\hat{x}, \hat{p})$ phase space. We follow the Claveles convention, where a positive angle rotates the $x$-quadrature toward the $p$-quadrature:

$$
\hat{R}(\theta) = \exp\!\left(i\theta\, \hat{a}^\dagger \hat{a}\right).
$$

The CV Fourier transform corresponds to a 90-degree rotation ($\theta = -\pi/2$) in phase space, implemented by the $\hat{R}(-\pi/2)$ gate. Under this operation, the two quadratures transform as follows:

$$
\hat{R}^\dagger(-\pi/2)\, \hat{x}\, \hat{R}(-\pi/2) = \hat{p}, \qquad
\hat{R}^\dagger(-\pi/2)\, \hat{p}\, \hat{R}(-\pi/2) = -\hat{x}.
$$

From these equations, one obtains

$$
    \begin{aligned} 
        \hat{x}\, \hat{R}(-\pi/2)\ket{p=t} &= \hat{R}(-\pi/2)\hat{R}^\dagger(-\pi/2)\, \hat{x}\, \hat{R}(-\pi/2)\ket{p=t}\\
        &= \hat{R}(-\pi/2)\, \hat{p}\ket{p=t} \\
        &= t\hat{R}(-\pi/2)\,\ket{p=t}
    \end{aligned}
$$

and thus $\hat{R}(-\pi/2)\ket{p=t} = \ket{x=t} \iff \bra{x=t} \hat{R}(-\pi/2) = \bra{p=t}$ follows. The phase rotation by $\theta = -\pi/2$ therefore directly maps the $x$-quadrature wave function to the Fourier transform of the original state. For $\ket{\phi} = \hat{R}(-\pi/2)\ket{\psi}$, its position-space wave function is represented as:

$$
    \phi(t) = \braket{x=t|\hat{R}(-\pi/2)|\psi} = \braket{p=t|\psi} = \tilde\psi(t)
$$

which directly implies

$$
    \phi(x) = \mathcal{F}[\psi](x).
$$

Consequently, the $x$-quadrature measurement distribution following $\hat{R}(-\pi/2)$ should agree with the Fourier-transformed distribution:

$$
P(x) = |\phi(x)|^2 = |\mathcal{F}[\psi](x)|^2.
$$

We verify this relationship below by computing the right-hand side analytically and comparing it against the simulated homodyne measurement of the left-hand side.

---

## 1. Setup

First, let us import the required libraries. In this tutorial we employ $\hbar = 1$ in accordance with the simulator convention; for a general value of $\hbar$, the argument below would be scaled to $p/\hbar$.

In [ ]:
from math import factorial, pi

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from claveles.circuit import CircuitRepr
from claveles.circuit.ops.std import MeasureHomodyne, PhaseRotation
from claveles.circuit.state import QuantumState
from claveles.client import SimulatorClient
from claveles.constant import hbar
from claveles.execute import execute
from matplotlib.colors import Normalize
from numpy.polynomial.hermite import hermval

---

## 2. Initial State and its Analytic Fourier Transform

### 2.1 Defining the Initial State

We select an equal superposition of the three lowest Fock states as our initial state:

$$
\ket{\psi} \propto \ket{0} + \ket{1} + \ket{2}.
$$

We store the amplitudes $c_n$ as a normalized state vector. A Fock number cutoff of three is exactly sufficient, as the phase rotation gate is the sole operation in this demonstration and remains diagonal in the Fock basis, guaranteeing the simulation to be exact without truncation errors.

In [ ]:
cutoff_dim = 3  # Exact: R(-pi/2) stays within |0>, |1>, |2>

coeffs = np.zeros(cutoff_dim, dtype=np.complex128)
coeffs[0:3] = 1.0
coeffs /= np.linalg.norm(coeffs)
print("amplitudes c_n:", np.round(coeffs, 3))

### 2.2 Closed-Form Wave Function and its Fourier Transform

The $x$-quadrature wave function of a Fock state $\ket{n}$ is given by the standard Hermite–Gauss function:

$$
\psi_n(x) = \frac{1}{\sqrt{2^n n!}}\,(\pi\hbar)^{-1/4}\,
            H_n\!\left(\frac{x}{\sqrt{\hbar}}\right)\,
            e^{-x^2/(2\hbar)},
$$

where $\hbar$ is explicitly included; the simulator evaluates at $\hbar = 1$. The overall wave function of our state is the closed-form sum $\psi(x) = \sum_n c_n \psi_n(x)$.

In [ ]:
def fock_wave_function(n, x):
    """x-quadrature wave function psi_n(x) of |n> (hbar units)."""
    coef = np.zeros(n + 1)
    coef[n] = 1.0
    hermite = hermval(x / np.sqrt(hbar), coef)
    norm = (1.0 / np.sqrt(2**n * factorial(n))) * (1.0 / (pi * hbar)) ** 0.25
    return norm * hermite * np.exp(-(x**2) / (2 * hbar))


def wave_function(coeffs, x):
    """x-quadrature wave function of sum_n coeffs[n] |n>."""
    return sum(coeffs[n] * fock_wave_function(n, x) for n in range(len(coeffs)))

The Hermite–Gauss functions serve as eigenfunctions of the Fourier transform operator, satisfying $\mathcal{F}[\psi_n] = (-i)^n \psi_n$. This aligns perfectly with the state rotation relation $\hat{R}(-\pi/2)\ket{n} = e^{-i(\pi/2)n}\ket{n} = (-i)^n\ket{n}$. Consequently, the Fourier transform of $\psi$ can be written in a closed form where each constituent term maps to an expanded phase factor of $(-i)^n$:

$$
\mathcal{F}[\psi](x) = \sum_n c_n\,(-i)^n\,\psi_n(x)
$$

which also implies that the wave functions of Fock eigenstates $\psi_n(x)$ comprise the family of eigenfunctions of the Fourier transform (each with an eigenvalue of $(-i)^n$).

In [ ]:
def fourier_wave_function(coeffs, x):
    """Fourier transform of the wave function, using F[psi_n] = (-i)^n psi_n."""
    return sum(
        coeffs[n] * (-1j) ** n * fock_wave_function(n, x) for n in range(len(coeffs))
    )

We plot the initial probability density $P(x) = |\psi(x)|^2$ alongside the expected post-rotation density $|\mathcal{F}[\psi](x)|^2$.

In [ ]:
grid = np.linspace(-6, 6, 601)
p_before = np.abs(wave_function(coeffs, grid)) ** 2
p_expected = np.abs(fourier_wave_function(coeffs, grid)) ** 2

sns.set_style("whitegrid")
plt.figure(figsize=(7, 4))
plt.plot(grid, p_before, label=r"$P(x) = |\psi(x)|^2$ (before)")
plt.plot(grid, p_expected, "--", label=r"$|\mathcal{F}[\psi](x)|^2$ (expected after)")
plt.xlabel("quadrature value")
plt.ylabel("probability density")
plt.title(r"$|0\rangle + |1\rangle + |2\rangle$ and its Fourier transform")
plt.legend()
plt.tight_layout()
plt.show()

The two profiles display distinctly different geometric shapes. Our objective is to reproduce the dashed curve experimentally through a homodyne measurement of the $x$-quadrature following the application of the $\hat{R}(-\pi/2)$ gate.

This transformation becomes even more intuitive when viewed in phase space. The Wigner function of $\hat{R}(-\pi/2)\ket{\psi}$ is simply the Wigner function of $\ket{\psi}$ rotated by 90 degrees. The two probability densities plotted above correspond to its projections onto the $x$-axis before and after this rotation. Given that $\hat{R}(-\pi/2)\ket{n} = (-i)^n\ket{n}$, the rotated state vector possesses transformed amplitudes $c_n (-i)^n$, which we assemble using `QuantumState.from_state_vector` and evaluate on a grid via `wigner_func`.

In [ ]:
# Wigner functions before and after the rotation. R(-pi/2)|n> = (-i)^n |n>,
# so the rotated state has amplitudes c_n (-i)^n.
rotated_coeffs = coeffs * np.array([(-1j) ** n for n in range(cutoff_dim)])
state_before = QuantumState.from_state_vector(coeffs, cutoff_dim)
state_after = QuantumState.from_state_vector(rotated_coeffs, cutoff_dim)

wvec = np.linspace(-5, 5, 201)
x_mesh, p_mesh = np.meshgrid(wvec, wvec)
w_before = state_before.wigner_func()(x_mesh, p_mesh)
w_after = state_after.wigner_func()(x_mesh, p_mesh)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, w, title in [
    (axes[0], w_before, r"Wigner function of $|\psi\rangle$"),
    (axes[1], w_after, r"Wigner function of $\hat{R}(-\pi/2)|\psi\rangle$"),
]:
    scale = float(np.max(np.abs(w)))
    ax.contourf(wvec, wvec, w, 60, cmap="RdBu", norm=Normalize(-scale, scale))
    ax.set_aspect("equal")
    ax.set_xlabel("x")
    ax.set_ylabel("p")
    ax.set_title(title)
plt.tight_layout()
plt.show()

The Wigner function clearly undergoes a rigid 90-degree rotation. Projecting these phase-space distributions onto the $x$-axis reconstructs the solid and dashed curves shown above, respectively.

---

## 3. Applying $\hat{R}(-\pi/2)$ and Measuring

### 3.1 Preparing the State for the Simulator

`QuantumState.from_state_vector` builds the input state directly from its state vector in the Fock-number basis, and the simulator holds it in the Fock representation with the cutoff given here. Accordingly, we pass the raw amplitudes $c_n$ defined above without applying additional approximations.

In [ ]:
initial_state = QuantumState.from_state_vector(coeffs, cutoff_dim)

### 3.2 Circuit and Measurement

We construct a quantum circuit that places this state on mode 0, applies the $\hat{R}(-\pi/2)$ gate, and subsequently executes a homodyne measurement of the $x$-quadrature over a large number of shots. The operation `PhaseRotation(phi)` corresponds exactly to $\hat{R}(\phi)$ under the convention detailed above, meaning the physical gate is declared via `PhaseRotation(phi=-np.pi/2)`. Since `MeasureHomodyne(phi)` evaluates the generalized quadrature operator $\hat{x}\cos\phi + \hat{p}\sin\phi$, sampling the base $x$-quadrature corresponds to setting `phi=0`.

In [ ]:
circuit = CircuitRepr("fourier")
circuit.Q(0) | initial_state
circuit.Q(0) | PhaseRotation(phi=-np.pi / 2)  # R(-pi/2)
circuit.Q(0) | MeasureHomodyne(phi=0.0)  # Measure x quadrature

client = SimulatorClient(backend="cpu", state_save_policy="none")
client.n_shots = 3000
result = execute(circuit, client)

samples = np.array(
    [result.client_result[i].get_value(0) for i in range(client.n_shots)]
)
print(f"measured x: mean = {samples.mean():.3f}, std = {samples.std():.3f}")

### 3.3 Comparing the Measurement against the Predicted Fourier Transform

Finally, we extract a histogram of the sampled outcomes $x$ and overlay the analytic prediction $|\mathcal{F}[\psi](x)|^2$ derived in Section 2.

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(
    samples,
    bins=50,
    range=(-6, 6),
    density=True,
    alpha=0.6,
    label=r"measured $x$ after $\hat{R}(-\pi/2)$",
)
plt.plot(
    grid, p_expected, "r-", linewidth=1.5, label=r"expected $|\mathcal{F}[\psi](x)|^2$"
)
plt.plot(
    grid,
    p_before,
    "k--",
    linewidth=1.0,
    label=r"original $|\psi(x)|^2$ (for reference)",
)
plt.xlabel("quadrature value")
plt.ylabel("probability density")
plt.title(r"Homodyne measurement after $\hat{R}(-\pi/2)$ vs. Fourier transform")
plt.legend()
plt.tight_layout()
plt.show()

The empirical histogram of the measured $x$ values shows an excellent match with the analytical Fourier transform curve $|\mathcal{F}[\psi](x)|^2$ (red curve), departing clearly from the initial $x$-quadrature distribution profile (dashed black curve). This provides robust verification that the $\hat{R}(-\pi/2)$ phase gate effectively implements the continuous Fourier transform: sampling the base $x$-quadrature post-rotation accurately reproduces the Fourier-transformed distribution of the input quantum state.

---

## Summary

In this tutorial, we have demonstrated the following:

1. **Phase rotation as a Fourier transform**: The $\hat{R}(-\pi/2)$ gate acts as a rigid 90-degree rotation in phase space that interchanges the principal quadrature operators ($\hat{x} \to \hat{p}$, $\hat{p} \to -\hat{x}$), mapping the output $x$-quadrature wave function directly onto the Fourier transform of the input wave function.
2. **Closed-form prediction**: For the state $\ket{\psi} \propto \ket{0}+\ket{1}+\ket{2}$, the total wave function is expanded as a finite sum of Hermite–Gauss functions. Since these functions serve as natural eigenfunctions of the Fourier transform with corresponding eigenvalues of $(-i)^n$ (matching the gate relation $\hat{R}(-\pi/2)\ket{n} = (-i)^n\ket{n}$), the exact analytical Fourier transform is obtained by mapping a discrete phase factor of $(-i)^n$ to each individual term.
3. **Verification by homodyne measurement**: By initializing the quantum state directly in the number basis via `QuantumState.from_state_vector`, applying a $\hat{R}(-\pi/2)$ rotation, and executing homodyne detection along the $x$-quadrature, we gather an empirical distribution that matches the analytically derived prediction $|\mathcal{F}[\psi](x)|^2$.